# 한국어 Kiwi와 BM25

이 노트북은 빈 Python 3.12 커널에서 순서대로 실행합니다.

여기서 검색 토큰은 Kiwi의 형태소 `Token.form`입니다. Ch5의 모델 입력 토큰이나 Ch10의 `cl100k_base`와 분할 규칙이 다르므로 토큰 수를 직접 비교하지 않습니다.


## 실행 환경 준비

Kiwi 0.23.2와 BM25 구현을 사용합니다. 외부 API는 호출하지 않습니다.


In [1]:
%pip install -q "kiwipiepy==0.23.2" "rank-bm25==0.2.2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

course_root = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "labs/part3/ch12-retrieval-tuning/data").is_dir()), None)
assert course_root is not None, "Ch 12 실습 데이터 폴더를 찾지 못했습니다."
chapter_dir = course_root / "labs/part3/ch12-retrieval-tuning"
data_dir = chapter_dir / "data"
docs = json.loads((data_dir / "core-documents.json").read_text())
queries = json.loads((data_dir / "core-queries.json").read_text())
kiwi = Kiwi()
# 내용어와 숫자·외국어를 남기는 수업용 정책입니다.
KEEP = ("N", "V", "VA", "MAG", "SN", "SL")
def kiwi_terms(text):
    return [token.form for token in kiwi.tokenize(text) if token.tag.startswith(KEEP)]

## 기본 실행

같은 한국어 문장을 공백과 형태소로 나눈 결과를 비교합니다.


In [3]:
sample = "예약자가 없으면 일반 도서를 연장할 수 있습니다."
print("공백 분리:", sample.split())
print("Kiwi 형태소:", kiwi_terms(sample))

공백 분리: ['예약자가', '없으면', '일반', '도서를', '연장할', '수', '있습니다.']
Kiwi 형태소: ['예약', '없', '일반', '도서', '연장', '수', '있']


## 중간 결과 관찰

문서와 질문에 같은 토큰 정책을 적용해 BM25 순위를 만듭니다.


In [4]:
corpus = [kiwi_terms(f"{d['title']} {d['text']}") for d in docs]
bm25 = BM25Okapi(corpus)
def sparse_rank(question, k=3):
    scores = bm25.get_scores(kiwi_terms(question))
    order = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:k]
    return [(docs[i]["doc_id"], round(float(scores[i]), 3)) for i in order]
print("질문:", queries[2]["question"])
print("질문 형태소:", kiwi_terms(queries[2]["question"]))
print("상위 3개:", sparse_rank(queries[2]["question"]))

질문: ISBN으로 없는 책을 신청하려면?
질문 형태소: ['ISBN', '없', '책', '신청']
상위 3개: [('LIB-003', 6.292), ('ACC-003', 1.48), ('LIB-004', 1.438)]


## 값 하나를 바꿔 비교

토큰 정책만 공백 분리로 바꾸고 같은 질문의 점수를 비교합니다.


In [5]:
space_corpus = [f"{d['title']} {d['text']}".split() for d in docs]
space_bm25 = BM25Okapi(space_corpus)
question = queries[1]["question"]
space_scores = space_bm25.get_scores(question.split())
space_order = sorted(range(len(docs)), key=lambda i: space_scores[i], reverse=True)[:3]
print("공백 순위:", [docs[i]["doc_id"] for i in space_order])
print("Kiwi 순위:", [doc_id for doc_id, _ in sparse_rank(question)])

공백 순위: ['LIB-001', 'LIB-002', 'LIB-003']
Kiwi 순위: ['LIB-002', 'LIB-003', 'LIB-004']


## 반복 실험과 결과 해석

8개 질문의 Recall@3를 계산합니다. 작은 자료에서 나온 값은 토큰 정책의 예시입니다.


In [6]:
rows = []
for query in queries:
    ranked = [doc_id for doc_id, _ in sparse_rank(query["question"])]
    rows.append({"qid": query["qid"], "ranked": ranked,
                 "hit": query["gold_doc_id"] in ranked})
print("Kiwi+BM25 Recall@3:", sum(row["hit"] for row in rows), "/", len(rows))
print("기록할 결과 행:", len(rows))

Kiwi+BM25 Recall@3: 8 / 8
기록할 결과 행: 8


<h3>실행 결과 보존</h3><p>기존 정식 결과를 해시 이름으로 보존하고, 이번 결과를 실행 시각이 포함된 파일에 먼저 저장한 뒤 최신 결과 파일을 갱신합니다.</p>

In [7]:
import hashlib
from datetime import UTC, datetime
payload = json.dumps(rows, ensure_ascii=False, indent=2)
runs = chapter_dir / "runs"; history = runs / "core-history"
history.mkdir(parents=True, exist_ok=True)
pointer = runs / "core-04-bm25.json"
if pointer.exists():
    old = pointer.read_bytes()
    backup = history / f"before-{hashlib.sha256(old).hexdigest()[:12]}-{pointer.name}"
    if not backup.exists(): backup.write_bytes(old)
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%f")
archive = history / f"{run_id}-{pointer.name}"
archive.write_text(payload, encoding="utf-8")
pointer.write_text(payload, encoding="utf-8")
print("실행별 보존 로그:", archive.relative_to(course_root))

실행별 보존 로그: labs/part3/ch12-retrieval-tuning/runs/core-history/20260909T175148132204-core-04-bm25.json
